# 6th attempt - RCNN - 5

In [84]:
# 1️⃣ ייבוא המודול והפונקציות
import functions         # ייבוא רגיל למודול
from functions import *  # ייבוא כל הפונקציות ל־namespace

# 2️⃣ טעינה מחדש
import importlib
importlib.reload(functions)  # מוודא שהמודול מעודכן

# 3️⃣ הרצת הפונקציות שוב ל־namespace
from functions import *  # שוב מייבא את כל הפונקציות אחרי ה־reload

In [85]:
import numpy as np
import pandas as pd
from functions import *
from read_from_file_df import *
import pickle
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [86]:
SIZE = 5
AMOUNT_BOARDS = 10000
AMOUNT_MOVES = 100
NUM_DICT = 1
gen = 2

In [87]:
reverse_df_sort = load_reverse_df(SIZE, AMOUNT_BOARDS, gen)
X_train, X_val, X_test, y_train, y_val, y_test = prepare_reverse_dataset(reverse_df_sort, SIZE, gen, target_pixel_index=0, test_size=0.1, val_size=0.1, random_state=365)
X_train_array, X_val_array, X_test_array, y_train_array, y_val_array, y_test_array = to_numpy_4d(X_train, X_val, X_test, y_train, y_val, y_test, SIZE)

print("len df:", len(reverse_df_sort))
print("len train: ", len(X_train))
print("len val: ",len(X_val))
print("len test: ",len(X_test))

len df: 29760
len train:  24105
len val:  2679
len test:  2976


In [88]:
X_train.shape

(24105, 25)

In [89]:
X_train_array, X_val_array, X_test_array, y_train_array, y_val_array, y_test_array = to_numpy_4d(X_train, X_val, X_test, y_train, y_val, y_test, SIZE)

print(X_train_array.shape)
print(y_train_array.shape)
print(X_val_array.shape)
print(y_val_array.shape)
print(X_test_array.shape)
print(y_test_array.shape)

(24105, 5, 5, 1)
(24105, 1)
(2679, 5, 5, 1)
(2679, 1)
(2976, 5, 5, 1)
(2976, 1)


In [90]:
import tensorflow as tf


def build_RCNN5_softmax(gen_data, X_train_array, y_train_array, SIZE, batch_size=32, epochs=3):
    # sourcery skip: inline-immediately-returned-variable
    # --- פרמטרים ---
    gen_data = gen - 1    # מספר הלוחות הרציפים בקלט
    BATCH_SIZE = batch_size
    EPOCHS = epochs

    # --- PREPROCESSING ---
    # X_train_array.shape = (num_samples + gen_data, SIZE, SIZE, 1)
    # y_train_array.shape = (num_samples, 1)  ← תא אחד בלבד (0 או 1)

    num_samples = X_train_array.shape[0] - gen_data

    X_train = np.zeros((num_samples, gen_data, SIZE, SIZE, 1), dtype='float32')
    y_train = np.zeros((num_samples, 1), dtype='float32')  # רק תא אחד

    # # Convert labels to one-hot (for softmax)
    # y_train = tf.keras.utils.to_categorical(y_train, num_classes=2)
    # y_val   = tf.keras.utils.to_categorical(y_val,   num_classes=2)
    # y_test  = tf.keras.utils.to_categorical(y_test,  num_classes=2)

    for i in range(num_samples):
        X_train[i] = X_train_array[i:i+gen_data].reshape(gen_data, SIZE, SIZE, 1)   # רצף של gen_data לוחות
        y_train[i] = y_train_array[i]              # הפלט: תא אחד (0/1)

    print("X_train shape:", X_train.shape)  # (num_samples, gen_data, SIZE, SIZE, 1)
    print("y_train shape:", y_train.shape)  # (num_samples, 1)

    # --- MODEL ---
    model = tf.keras.Sequential([
        tf.keras.layers.ConvLSTM2D(
            filters=32,
            kernel_size=(3,3),
            activation='relu',
            padding='same',
            return_sequences=True,
            input_shape=(gen_data, SIZE, SIZE, 1)
        ),
        tf.keras.layers.BatchNormalization(),

        tf.keras.layers.ConvLSTM2D(
            filters=64,
            kernel_size=(3,3),
            activation='relu',
            padding='same',
            return_sequences=False
        ),
        tf.keras.layers.BatchNormalization(),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    return model

In [91]:
model = build_RCNN5_softmax(gen_data=gen-1, X_train_array=X_train_array, y_train_array=y_train_array, SIZE=SIZE, batch_size=32, epochs=3)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

X_train shape: (24104, 1, 5, 5, 1)
y_train shape: (24104, 1)


c:\Users\דרור\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_34"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_lstm2d_68 (ConvLSTM2D)     │ (None, 1, 5, 5, 32)    │        38,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_68          │ (None, 1, 5, 5, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_lstm2d_69 (ConvLSTM2D)     │ (None, 5, 5, 64)       │       221,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_69          │ (None, 5, 5, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_34 (Flatten)            │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_68 (Dense)                │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_69 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 362,497 (1.38 MB)

 Trainable params: 362,305 (1.38 MB)

 Non-trainable params: 192 (768.00 B)

In [92]:
# ***** build RNN/RCNN train/test arrays *****
gen_data = gen - 1
num_samples_train = X_train_array.shape[0] - gen_data

X_train_rnn = np.zeros((num_samples_train, gen_data, SIZE, SIZE, 1), dtype='float32')
y_train_rnn = np.zeros((num_samples_train,), dtype='float32')
for i in range(num_samples_train):
    X_train_rnn[i] = X_train_array[i:i + gen_data].reshape(gen_data, SIZE, SIZE, 1)
    y_train_rnn[i] = y_train_array[i].astype('float32')

# אימון
history = model.fit(
    X_train_rnn,
    y_train_rnn,
    validation_split=0.2,
    epochs=10,
    batch_size=32,
    shuffle=True
)

C:\Users\דרור\AppData\Local\Temp\ipykernel_11300\1525982118.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  y_train_rnn[i] = y_train_array[i].astype('float32')


Epoch 1/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 21s 21ms/step - accuracy: 0.6633 - loss: 0.6301 - val_accuracy: 0.6716 - val_loss: 0.6133
Epoch 2/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.7084 - loss: 0.5764 - val_accuracy: 0.6866 - val_loss: 0.5943
Epoch 3/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - accuracy: 0.7147 - loss: 0.5555 - val_accuracy: 0.7009 - val_loss: 0.5850
Epoch 4/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 18s 18ms/step - accuracy: 0.7211 - loss: 0.5447 - val_accuracy: 0.6907 - val_loss: 0.5831
Epoch 5/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - accuracy: 0.7471 - loss: 0.5143 - val_accuracy: 0.6953 - val_loss: 0.6045
Epoch 6/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - accuracy: 0.7681 - loss: 0.4833 - val_accuracy: 0.6911 - val_loss: 0.6247
Epoch 7/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 21s 19ms/step - accuracy: 0.7905 - loss: 0.4456 - val_accuracy: 0.6841 - val_loss: 0.6953
Epoch 8/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.8209 - loss: 0.3942 - 

In [93]:
num_samples_test = X_test_array.shape[0] - gen_data
X_test = np.zeros((num_samples_test, gen_data, SIZE, SIZE, 1), dtype='float32')
y_test = np.zeros((num_samples_test,), dtype='float32')
for i in range(num_samples_test):
    X_test[i] = X_test_array[i:i+gen_data].reshape(gen_data, SIZE, SIZE, 1)
    y_test[i] = y_test_array[i]

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc:.3f}")

C:\Users\דרור\AppData\Local\Temp\ipykernel_11300\2954292026.py:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  y_test[i] = y_test_array[i]


93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.6441 - loss: 0.8702
Test accuracy: 0.657


In [94]:
import matplotlib.pyplot as plt
import numpy as np

def plot_results(y_test, y_pred):
    """
    מצייר פיזור בין הפלט האמיתי לפלט החזוי.
    """
    y_test = y_test.flatten()
    y_pred = y_pred.flatten()

    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, alpha=0.3)
    
    # קו אידיאלי (תחזית = אמת)
    plt.plot([0, 1], [0, 1], linestyle='--')

    plt.xlabel("True Value (y_test)")
    plt.ylabel("Predicted Value (y_pred)")
    plt.title("Prediction Scatter Plot")
    plt.grid(True)
    plt.show()


In [95]:
# יצירת סט בדיקה
num_samples_test = X_test_array.shape[0] - gen_data
X_test = np.zeros((num_samples_test, gen_data, SIZE, SIZE, 1), dtype='float32')
y_test = np.zeros((num_samples_test, 1), dtype='float32')

for i in range(num_samples_test):
    X_test[i] = X_test_array[i:i+gen_data].reshape(gen_data, SIZE, SIZE, 1)
    y_test[i] = y_test_array[i]

# הפעלת פונקציית ההערכה
y_test_flat = y_test.flatten()
evaluate_model(model, X_test, y_test_flat)


93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step

===== Evaluation Results =====
┌──────────────┬────────────┬────────────┐
│              │ Pred=Alive │ Pred=Dead  │
├──────────────┼────────────┼────────────┤
│ True=Alive   │        417 │        604 │
│ True=Dead    │        415 │       1539 │
└──────────────┴────────────┴────────────┘

--- Performance Metrics ---
Accuracy    : 0.657
Precision   : 0.501
Recall      : 0.408
F1-score    : 0.450


In [96]:
amount_features = len(reverse_df_sort.columns) - SIZE*SIZE #the previous boards
features = reverse_df_sort.iloc[:, :amount_features]
for i in range(SIZE*SIZE): # to any pixel in the expected board
    name_col = 'Col_' + str(i+amount_features)
    target = reverse_df_sort[name_col]
    X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.1, random_state=365)
    
    X_train_array = X_train.to_numpy()
    y_train_array = y_train.to_numpy()
    X_train_array = X_train_array.reshape((X_train.shape[0],SIZE,SIZE,1))
    y_train_array = y_train_array.reshape((y_train.shape[0],1))
    
    model = build_RCNN5_softmax(gen_data, X_train_array, y_train_array, SIZE, 32, 3)
    name_file = f"{PATH_MODELS}\\model6_RCNN_softmax\{SIZE}\\size_{SIZE}_pixel_{str(i+1)}.pkl"
    with open(name_file, 'wb') as file:
        pickle.dump(model, file)
    print(i)

X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)


c:\Users\דרור\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


0
X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
1
X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
2
X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
3
X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
4
X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
5
X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
6
X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
7
X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
8
X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
9
X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
10
X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
11
X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
12
X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
13
X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
14
X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
15
X_train shape: (26783, 1, 5, 5, 1)
y_train shap